# Histogram and LASSO feature preparation


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
while not (PROJECT_ROOT / "modeling_pipeline.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this code from the MMDLPC_Code_PDF folder.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parent / "MMDLPC_Code_PDF_Data"
OUTPUT_DIR = PROJECT_ROOT / '03_Pathomics/00_Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_INPUT_DIR = Path(os.environ.get('MMDLPC_EXTERNAL_INPUTS', str(DATA_ROOT / 'External_Inputs')))

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
sys.path.insert(0, str(PROJECT_ROOT))
from modeling_pipeline import (
    read_indexed, parse_p_value, correlation_keep_indices, make_pipeline,
    training_cv, best_finite_parameters, fit_on_training,
    model_estimators, training_search_grids, positive_probability,
)
DATA_DIR = DATA_ROOT / '00_Shared_Data_and_Code/Data'


## Histogram LASSO


In [ ]:
probability_file = DATA_DIR / 'superwise_path_prob_histogram.csv'
prediction_file = DATA_DIR / 'superwise_path_pred_histogram.csv'


In [ ]:
prob_histo = read_indexed(probability_file)
pred_histo = read_indexed(prediction_file)
if set(prob_histo.index) != set(pred_histo.index):
    raise ValueError('The pathology histograms must contain the same patient IDs.')
pathology_features = prob_histo.join(pred_histo, how='inner', validate='one_to_one')

partition = read_indexed(DATA_DIR / 'P_fixed_partition.csv')
if set(partition.Split) != {'Train', 'Test'}:
    raise ValueError('Expected a pre-specified Train/Test partition.')
label_table = read_indexed(DATA_DIR / 'CPGEA-TCGA 20230106 OK.csv')
if not partition.index.isin(pathology_features.index).all() or not partition.index.isin(label_table.index).all():
    raise ValueError('The feature or label table is missing patients in the partition.')
raw_features = pathology_features.loc[partition.index].apply(pd.to_numeric, errors='raise')
raw_features = raw_features.replace([np.inf, -np.inf], np.nan)
X_source = raw_features.add_prefix('pathology__')
y_source = label_table.loc[partition.index, 'HRR_ANY']
if not y_source.isin([0, 1]).all():
    raise ValueError('Expected observed binary HRR_ANY labels.')
y_source = y_source.astype(int)
split = partition.Split
train_rows = split.eq('Train')
test_rows = split.eq('Test')
train_ids = X_source.index[train_rows]
test_ids = X_source.index[test_rows]
if not X_source.index.is_unique or not set(train_ids).isdisjoint(test_ids):
    raise ValueError('Patient IDs must be unique and the partition must be disjoint.')
labels = ['HRR_ANY']
label_data = label_table.loc[partition.index, labels + ['group']].reset_index()
ids = pd.Series(partition.index, index=partition.index, name='ID')

if not label_table.loc[test_ids, 'group'].eq('CPGEA').all() or label_table.loc[train_ids, 'group'].eq('CPGEA').any():
    raise ValueError('The external-cohort partition is inconsistent with the clinical labels.')
if raw_features.isna().any().any() or (raw_features < 0).any().any():
    raise ValueError('Pathology frequencies must be observed and nonnegative.')
structed_data = raw_features.join(label_table.loc[partition.index, labels + ['group']], validate='one_to_one')


In [ ]:
structed_data.columns


In [ ]:
ids = pd.Series(partition.index, index=partition.index, name='ID')


In [ ]:
structed_data.describe()


In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
features = raw_features.copy()
tfidf_models = {}
for kind in ['prob', 'pred']:
    columns = [column for column in raw_features if column.startswith(kind + '-')]
    transformer = TfidfTransformer().fit(raw_features.loc[train_rows, columns])
    tfidf_models[kind] = transformer
    names = [column.replace('-', '').replace('.', '') for column in columns]
    values = pd.DataFrame(transformer.transform(raw_features[columns]).toarray(),
                          index=raw_features.index, columns=names)
    features = features.join(values, validate='one_to_one')
feature_columns = features.columns.tolist()
data = features.join(label_table.loc[partition.index, labels + ['group']], validate='one_to_one')
data.describe()


In [ ]:
spearman_corr = data.loc[train_rows, feature_columns].corr('spearman')


In [ ]:
variable_columns = [column for column in feature_columns if data.loc[train_rows, column].nunique() > 1]
positions = correlation_keep_indices(data.loc[train_rows, variable_columns].to_numpy(), 'spearman', .9)
sel_feature = [variable_columns[i] for i in positions]


In [ ]:
sel_data = data[sel_feature + labels + ['group']]
sel_data.columns


In [ ]:
X_data = X_source.loc[train_rows].copy()
X_test_data = X_source.loc[test_rows].copy()
y_data = y_source.loc[train_rows].to_frame('HRR_ANY')
y_test_data = y_source.loc[test_rows].to_frame('HRR_ANY')
n_classes = 2


In [ ]:
selection_classifier = LogisticRegression(max_iter=1000, random_state=0)
selection_grid = training_search_grids('P', {'LR': selection_classifier})['LR']
selection_search = GridSearchCV(
    make_pipeline(X_data, 'P', selection_classifier), selection_grid,
    scoring='roc_auc', cv=training_cv(y_data['HRR_ANY']),
    n_jobs=1, error_score=np.nan, refit=False)
selection_search.fit(X_data, y_data['HRR_ANY'])
selection_parameters = best_finite_parameters(selection_search)
selection_pipeline = make_pipeline(X_data, 'P', selection_classifier)
selection_pipeline.set_params(**selection_parameters).fit(X_data, y_data['HRR_ANY'])
selector = selection_pipeline['features'].named_transformers_['P']
alpha = selector.alpha


In [ ]:
selection_results = pd.DataFrame(selection_search.cv_results_)
alpha_key = next(key for key in selection_results if key.endswith('__alpha'))
plt.figure(figsize=(6, 4))
plt.errorbar(selection_results[alpha_key].astype(float), selection_results['mean_test_score'],
             yerr=selection_results['std_test_score'], marker='o', markersize=3)
plt.xscale('log')
plt.xlabel('LASSO alpha')
plt.ylabel('Training cross-validation AUC (mean and SD)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'Reference_Path_feature_selection_CV_AUC.pdf', bbox_inches='tight')


In [ ]:
models = [selector.lasso_]
column_names = np.asarray(selector.observed_columns_)[selector.correlation_indices_]


In [ ]:
selected_features = [selector.selected_features_.tolist()]
feat_coef = [(name, coefficient) for name, coefficient in zip(column_names, selector.lasso_.coef_)
             if abs(coefficient) > 1e-6]
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df


In [ ]:
feat_coef = sorted(feat_coef, key=lambda x: x[1])
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df.plot(x='feature_name', y='Coefficients', kind='barh')

plt.savefig(str(OUTPUT_DIR / f'Reference_Path_feature_weights.svg'), bbox_inches = 'tight')
plt.savefig(str(OUTPUT_DIR / f'Reference_Path_feature_weights.pdf'), bbox_inches = 'tight')


In [ ]:
selected_values = pd.DataFrame(selection_pipeline['features'].transform(X_source),
    index=X_source.index, columns=selection_pipeline['features'].get_feature_names_out())
selected_values.index.name = 'ID'
selected_values.reset_index().to_csv(OUTPUT_DIR / 'path_sel_features_reference.csv', index=False)
selected_values.columns
